# English FastSpeech 2 (Svarah) — Kaggle

IIIT-H Speech Processing Lab. Deadline Saturday 7 November 2026, midnight IST.

Train the **English** model only. Telugu is the other notebook. This notebook does not ship audio or metrics; it produces them on the GPU.

**Before Run all:**

1. Accept https://huggingface.co/datasets/ai4bharat/Svarah while logged into the Hugging Face account that owns the token.
2. Notebook settings: **Accelerator → GPU T4**, **Internet → On**.
3. **Add-ons → Secrets**: add `HF_TOKEN` (a read token from https://huggingface.co/settings/tokens). The cells never print it.


## What this run does

Download a ~5 h / ~1 h subset, CTC-align it, extract pitch / energy / duration / mel, train up to 20,000 steps with a checkpoint every 2,000 steps under `English/checkpoints` (inside `/kaggle/working`), then write loss plots, the five test wavs, and the variance-adaptor plots into `Results/English/`.

If the session stops, run the notebook again. Training resumes from the newest checkpoint. To continue from a previous session, add that session's output as input and uncomment `FS2_EXTRA_CKPT` in the train cell.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/flyme2mars/iiith-fastspeech2-submission.git"
BRANCH = "main"
on_kaggle = Path("/kaggle/working").exists()
dest = Path("/kaggle/working/iiith-fastspeech2-assignment") if on_kaggle else Path.cwd()

if not on_kaggle and not (dest / "English" / "train.py").exists() and (Path.cwd() / "English" / "train.py").exists():
    dest = Path.cwd()

if on_kaggle:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        !git clone {REPO_URL} {dest}
    %cd {dest}
    !git fetch origin {BRANCH}
    !git checkout -B {BRANCH} origin/{BRANCH}
    !git reset --hard origin/{BRANCH}
else:
    %cd {dest}
    print("Not on Kaggle; leaving the git checkout alone. Use branch", BRANCH)

print("cwd", Path.cwd())
import torch
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
elif not os.environ.get("FS2_ALLOW_CPU"):
    raise SystemExit("Turn on Accelerator -> GPU T4 before running the rest of this notebook.")


In [ ]:
# Leave Kaggle's CUDA torch/torchaudio installed. requirements.txt would replace them.
!pip install -q -r requirements-kaggle.txt


In [ ]:
import os
os.environ.pop("HF_TOKEN", None)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets. The value is not printed.")
except Exception as exc:
    print("HF_TOKEN was not loaded:", type(exc).__name__)
    print("Svarah is gated. Add-ons -> Secrets -> label HF_TOKEN.")
    print("Accept the dataset first: https://huggingface.co/datasets/ai4bharat/Svarah")
    print("Telugu (IndicTTS) does not need this secret.")


## Download Svarah

Needs `HF_TOKEN` and a prior click on Accept. The script records the real hours in `data/english/split_stats.json`. Svarah's hub split is named `test`; the script builds its own train/val split from it.


In [ ]:
!python scripts/download_svarah.py


## Align and extract features

Default aligner is CTC (`facebook/wav2vec2-base-960h`), then pyworld pitch and STFT energy. This is the long CPU/GPU step before training. Add `--aligner mfa` only if Montreal Forced Aligner is already installed.


In [ ]:
!python scripts/align.py --lang english --aligner ctc
!python scripts/extract_features.py --lang english


In [ ]:
!python scripts/download_vocoder.py


## Train

Same step budget as the Telugu notebook (20,000, checkpoint every 2,000). If the T4 OOMs, set batch size 8 and `grad_acc_step` 2 in `configs/english.yaml` and resume.

The step-1 crash `Found dtype Double but expected Float` happened before any checkpoint was written. After this code is on `main`, start again at the default `restore_step` 0. Keep `preprocessed_data/english` (3502 train / 682 val). You can skip the download and align cells and run this train cell.


In [ ]:
import os
# os.environ["FS2_EXTRA_CKPT"] = "/kaggle/input/<previous-output>/English/checkpoints"
!python scripts/train.py --lang english


In [ ]:
!python scripts/plot_losses.py --lang english
!python scripts/synthesize.py --lang english
!python scripts/plot_variance.py --lang english


## Files to bring back

- `English/checkpoints/*.pth.tar` (gitignored; use Git LFS or a link in the README if they exceed 100 MB)
- `Results/English/training_loss.png`, `validation_loss.png`
- `Results/English/test_generated_audios/{E1,E2a,E2b,E3,E4}.wav`
- `Results/English/variance_adaptor_plots/{duration,pitch,energy}/*.png`
- `data/english/split_stats.json` for the report tables

Fill `report/REPORT.md` from those files. Do not invent the numbers.


In [ ]:
from pathlib import Path
root = Path("Results/English")
for path in sorted(root.rglob("*")):
    if path.is_file() and path.suffix != ".npz":
        print(path)
